In [3]:
import pandas as pd
from IPython.display import HTML
from rdkit import Chem, RDLogger
from rdkit.Chem import PandasTools
from chembl_webresource_client.new_client import new_client

RDLogger.DisableLog("rdApp.*")

TARGET_CHEMBL_ID = "CHEMBL3192"
TEST_SIZE = 0.2
STD_THRESHOLD = 0.5
ASSAY_CHUNK_SIZE = 50

# Search for target protein
Retrieve HDAC8 (`CHEMBL3192`) **binding** assays (`assay_type='B'`) with `confidence_score=9`, then collect IC50 activities for those assays.

In [4]:
assay_client = new_client.assay

assays = pd.DataFrame(
    assay_client.filter(
        target_chembl_id=TARGET_CHEMBL_ID,
        confidence_score=9,
        assay_type="B",
    ).only(
        [
            "assay_chembl_id",
            "target_chembl_id",
            "assay_type",
            "confidence_score",
            "confidence_description",
        ]
    )
)

print("Number of matching assays:", len(assays))
assays.head()

Number of matching assays: 882


,assay_chembl_id,assay_type,confidence_description,confidence_score,description,target_chembl_id
0,CHEMBL695893,B,Direct single protein target assigned,9,Inhibition of Histone deacetylase 8 (HDAC8) of...,CHEMBL3192
1,CHEMBL827903,B,Direct single protein target assigned,9,Inhibition of human histone deacetylase 8 prep...,CHEMBL3192
2,CHEMBL907065,B,Direct single protein target assigned,9,Inhibition of HDAC8 (mean IC50),CHEMBL3192
3,CHEMBL912072,B,Direct single protein target assigned,9,Inhibition of human HDAC8,CHEMBL3192
4,CHEMBL890787,B,Direct single protein target assigned,9,Inhibition of HDAC8 in HeLa cells,CHEMBL3192


In [5]:
assay_ids = assays["assay_chembl_id"].tolist()
activity_client = new_client.activity
activity_chunks = []

for start in range(0, len(assay_ids), ASSAY_CHUNK_SIZE):
    chunk_ids = assay_ids[start:start + ASSAY_CHUNK_SIZE]
    records = activity_client.filter(
        target_chembl_id=TARGET_CHEMBL_ID,
        standard_type="IC50",
        assay_chembl_id__in=chunk_ids,
    )
    activity_chunks.append(pd.DataFrame(records))

activities = pd.concat(activity_chunks, ignore_index=True)
activities = activities.merge(
    assays[["assay_chembl_id", "confidence_score", "confidence_description"]],
    on="assay_chembl_id",
    how="left",
)

print("Number of IC50 activity records:", len(activities))
print("\nAssay type distribution:")
print(assays["assay_type"].value_counts(dropna=False))
print("\nConfidence score distribution:")
print(assays["confidence_score"].value_counts(dropna=False))
print("\nStandard type:")
print(activities["standard_type"].value_counts(dropna=False))
print("All activities map to filtered assays:", activities["assay_chembl_id"].isin(assay_ids).all())
print("Target ChEMBL IDs:", activities["target_chembl_id"].unique())
activities[["assay_chembl_id", "confidence_score", "confidence_description"]].head()

Number of IC50 activity records: 4520

Assay type distribution:
assay_type
B    882
Name: count, dtype: int64

Confidence score distribution:
confidence_score
9    882
Name: count, dtype: int64

Standard type:
standard_type
IC50    4520
Name: count, dtype: int64
All activities map to filtered assays: True
Target ChEMBL IDs: ['CHEMBL3192']


,assay_chembl_id,confidence_score,confidence_description
0,CHEMBL695893,9,Direct single protein target assigned
1,CHEMBL695893,9,Direct single protein target assigned
2,CHEMBL695893,9,Direct single protein target assigned
3,CHEMBL695893,9,Direct single protein target assigned
4,CHEMBL695893,9,Direct single protein target assigned


# Handling data

Keep records with a numeric pChEMBL value and a valid, non-mixture SMILES string. Then aggregate replicate measurements of the same compound.

In [6]:
def smiles_to_inchi(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    inchi = Chem.MolToInchi(mol)
    return inchi if inchi else None


activities_clean = activities.loc[
    activities["canonical_smiles"].notna()
].copy()
activities_clean["pchembl_value"] = pd.to_numeric(
    activities_clean["pchembl_value"], errors="coerce"
)
activities_clean = activities_clean.dropna(subset=["pchembl_value"])
activities_clean = activities_clean[
    ~activities_clean["canonical_smiles"].str.contains(".", regex=False)
].copy()
activities_clean["inchi"] = activities_clean["canonical_smiles"].map(smiles_to_inchi)
activities_clean = activities_clean.dropna(subset=["inchi"])

print("Records after cleaning:", len(activities_clean))
print("Unique compounds before aggregation:", activities_clean["canonical_smiles"].nunique())
activities_clean.head()

Records after cleaning: 3390
Unique compounds before aggregation: 2661


,action_type,activity_comment,activity_id,activity_properties,assay_chembl_id,assay_description,assay_type,assay_variant_accession,assay_variant_mutation,bao_endpoint,...,text_value,toid,type,units,uo_units,upper_value,value,confidence_score,confidence_description,inchi
2,None,None,1270391,[],CHEMBL695893,Inhibition of Histone deacetylase 8 (HDAC8) of...,B,None,None,BAO_0000190,...,None,None,IC50,uM,UO_0000065,None,0.8,9,Direct single protein target assigned,InChI=1S/C27H27N5O4/c33-18-32(36)15-3-1-2-8-23...
3,None,None,1271622,[],CHEMBL695893,Inhibition of Histone deacetylase 8 (HDAC8) of...,B,None,None,BAO_0000190,...,None,None,IC50,uM,UO_0000065,None,0.69,9,Direct single protein target assigned,InChI=1S/C27H27N5O4/c33-18-32(36)13-7-1-2-10-2...
4,None,None,1275487,[],CHEMBL695893,Inhibition of Histone deacetylase 8 (HDAC8) of...,B,None,None,BAO_0000190,...,None,None,IC50,uM,UO_0000065,None,6.8,9,Direct single protein target assigned,InChI=1S/C13H18N2O3/c16-11-15(18)10-6-2-5-9-13...
5,None,None,1275492,[],CHEMBL695893,Inhibition of Histone deacetylase 8 (HDAC8) of...,B,None,None,BAO_0000190,...,None,None,IC50,uM,UO_0000065,None,0.78,9,Direct single protein target assigned,InChI=1S/C19H24N2O3/c22-15-21(24)13-7-3-1-2-4-...
6,None,None,1276796,[],CHEMBL695893,Inhibition of Histone deacetylase 8 (HDAC8) of...,B,None,None,BAO_0000190,...,None,None,IC50,uM,UO_0000065,None,9.7,9,Direct single protein target assigned,InChI=1S/C15H22N2O3/c18-13-17(20)12-6-2-5-11-1...


In [7]:
compounds = (
    activities_clean.groupby("canonical_smiles", as_index=False)
    .agg(
        molecule_chembl_id=("molecule_chembl_id", "first"),
        pchembl_value_mean=("pchembl_value", "mean"),
        pchembl_value_std=("pchembl_value", "std"),
        inchi=("inchi", "first"),
    )
)
compounds["pchembl_value_std"] = compounds["pchembl_value_std"].fillna(0)
compounds = compounds[
    [
        "molecule_chembl_id",
        "canonical_smiles",
        "pchembl_value_mean",
        "pchembl_value_std",
        "inchi",
    ]
]

print("Unique compounds:", len(compounds))
compounds.head()

Unique compounds: 2661


,molecule_chembl_id,canonical_smiles,pchembl_value_mean,pchembl_value_std,inchi
0,CHEMBL4569890,Brc1cncs1,5.25,0.0,InChI=1S/C3H2BrNS/c4-3-1-5-2-6-3/h1-2H
1,CHEMBL5177240,C#CCN(C)CCCOc1cc(NC(=O)CCCCCCC(=O)NO)c(Cl)cc1Cl,5.75,0.0,InChI=1S/C21H29Cl2N3O4/c1-3-11-26(2)12-8-13-30...
2,CHEMBL5170598,C#CCN(C)CCCOc1cc(NC(=O)CCCCCCC(=O)NO)ccc1Cl,7.30,0.0,InChI=1S/C21H30ClN3O4/c1-3-13-25(2)14-8-15-29-...
3,CHEMBL5187984,C#CCN(C)CCCOc1cc(NC(=O)CCCCCCCC(=O)NO)ccc1Cl,5.95,0.0,InChI=1S/C22H32ClN3O4/c1-3-14-26(2)15-9-16-30-...
4,CHEMBL5200566,C#CCN(C)CCCOc1cc(NC(=O)CCCCCCCCC(=O)NO)ccc1Cl,5.61,0.0,InChI=1S/C23H34ClN3O4/c1-3-15-27(2)16-10-17-31...


# Save to csv

In [8]:
compounds.to_csv("HDAC8_exp_data_inchi.csv", sep=",", index=False)

# Processing for QSAR

In [9]:
dataset = compounds.loc[compounds["pchembl_value_std"] < STD_THRESHOLD].copy()
dataset = dataset.drop(columns=["pchembl_value_std"]).reset_index(drop=True)

print("Compounds with pChEMBL SD <", STD_THRESHOLD, ":", len(dataset))
dataset.describe()

Compounds with pChEMBL SD < 0.5 : 2616


,pchembl_value_mean
count,2616.000000
mean,6.049619
std,0.885126
min,4.010000
25%,5.460000
50%,6.000000
75%,6.620000
max,9.990000


# Creating training and test samples
Use an **activity-stratified split** (test fraction = 0.2). Compounds are sorted by pChEMBL, then every *n*-th compound is assigned to the test set so that the activity range is represented in both splits.

In [10]:
def activity_stratified_split(df, activity_col="pchembl_value_mean", test_size=0.2):
    step = max(int(round(1.0 / test_size)), 2)
    df_sorted = df.sort_values(by=activity_col).reset_index(drop=True)
    data_test = df_sorted.iloc[::step].copy()
    data_train = df_sorted.drop(data_test.index).copy()
    return data_train.reset_index(drop=True), data_test.reset_index(drop=True)


data_train, data_test = activity_stratified_split(
    dataset,
    activity_col="pchembl_value_mean",
    test_size=TEST_SIZE,
)

In [11]:
print(f"Training compounds: {len(data_train)} ({len(data_train) / len(dataset):.1%})")
print(f"Test compounds: {len(data_test)} ({len(data_test) / len(dataset):.1%})")
print(
    "Training pChEMBL mean ± SD: "
    f"{data_train['pchembl_value_mean'].mean():.3f} ± {data_train['pchembl_value_mean'].std():.3f}"
    f"  (range {data_train['pchembl_value_mean'].min():.2f}–{data_train['pchembl_value_mean'].max():.2f})"
)
print(
    "Test pChEMBL mean ± SD: "
    f"{data_test['pchembl_value_mean'].mean():.3f} ± {data_test['pchembl_value_mean'].std():.3f}"
    f"  (range {data_test['pchembl_value_mean'].min():.2f}–{data_test['pchembl_value_mean'].max():.2f})"
)
data_train.head()

Training compounds: 2092 (80.0%)
Test compounds: 524 (20.0%)
Training pChEMBL mean ± SD: 6.049 ± 0.883  (range 4.02–9.92)
Test pChEMBL mean ± SD: 6.051 ± 0.895  (range 4.01–9.99)


,molecule_chembl_id,canonical_smiles,pchembl_value_mean,inchi
0,CHEMBL1767046,CNC(=O)/C(CCCCC[C@H](NCc1ccc(OC)cc1)C(=O)Nc1cc...,4.02,InChI=1S/C24H32N4O4/c1-25-23(29)22(28-31)12-8-...
1,CHEMBL1469,O=C(O)CCCc1ccccc1,4.03,InChI=1S/C10H12O2/c11-10(12)8-4-7-9-5-2-1-3-6-...
2,CHEMBL3605493,Nc1ccccc1NC(=O)c1ccc(CSC2=N[C@@H](Cc3ccccc3)CO...,4.06,InChI=1S/C24H23N3O2S/c25-21-8-4-5-9-22(21)27-2...
3,CHEMBL1767047,CNC(=O)/C(CCCCC[C@@H](C(=O)Nc1ccccc1)N(Cc1cccc...,4.06,InChI=1S/C30H36N4O3/c1-31-29(35)27(33-37)20-12...
4,CHEMBL3605499,Nc1ccccc1NC(=O)c1ccc(CNC2=N[C@@H](c3ccc(O)cc3)...,4.08,InChI=1S/C23H22N4O3/c24-19-3-1-2-4-20(19)26-22...


# Creating sdf files
The training set is written to `HDAC8_train.sdf` and the test set to `HDAC8_test.sdf`.

## Test set

In [12]:
PandasTools.AddMoleculeColumnToFrame(data_test, "canonical_smiles", "Molecule")
HTML(data_test.head().to_html())

,molecule_chembl_id,canonical_smiles,pchembl_value_mean,inchi,Molecule
0,CHEMBL5429446,Nc1ccccc1NC(=O)c1ccc(NC(=O)Cc2cccc3ccccc23)cn1,4.01,"InChI=1S/C24H20N4O2/c25-20-10-3-4-11-21(20)28-24(30)22-13-12-18(15-26-22)27-23(29)14-17-8-5-7-16-6-1-2-9-19(16)17/h1-13,15H,14,25H2,(H,27,29)(H,28,30)",<rdkit.Chem.rdchem.Mol object at 0x000001308F19A810>
1,CHEMBL3605494,Nc1ccccc1NC(=O)c1ccc(CNC2=N[C@@H](c3ccccc3)CO2)cc1,4.06,"InChI=1S/C23H22N4O2/c24-19-8-4-5-9-20(19)26-22(28)18-12-10-16(11-13-18)14-25-23-27-21(15-29-23)17-6-2-1-3-7-17/h1-13,21H,14-15,24H2,(H,25,27)(H,26,28)/t21-/m1/s1",<rdkit.Chem.rdchem.Mol object at 0x000001308F2CD2A0>
2,CHEMBL5419832,Nc1ccccc1NC(=O)c1ccc(NC(=O)Cc2ccccc2)cn1,4.11,"InChI=1S/C20H18N4O2/c21-16-8-4-5-9-17(16)24-20(26)18-11-10-15(13-22-18)23-19(25)12-14-6-2-1-3-7-14/h1-11,13H,12,21H2,(H,23,25)(H,24,26)",<rdkit.Chem.rdchem.Mol object at 0x000001308F2CCAC0>
3,CHEMBL5571764,CN(C(=O)c1ccc(-c2noc(C(F)(F)F)n2)cc1)c1ccccc1,4.14,"InChI=1S/C17H12F3N3O2/c1-23(13-5-3-2-4-6-13)15(24)12-9-7-11(8-10-12)14-21-16(25-22-14)17(18,19)20/h2-10H,1H3",<rdkit.Chem.rdchem.Mol object at 0x000001308F2CC3C0>
4,CHEMBL3771210,O=C(NO)c1csc(-c2cnccn2)n1,4.21,"InChI=1S/C8H6N4O2S/c13-7(12-14)6-4-15-8(11-6)5-3-9-1-2-10-5/h1-4,14H,(H,12,13)",<rdkit.Chem.rdchem.Mol object at 0x000001308F2CC510>


In [13]:
sdf_properties = [col for col in data_test.columns if col != "Molecule"]
PandasTools.WriteSDF(
    data_test,
    "HDAC8_test.sdf",
    molColName="Molecule",
    properties=sdf_properties,
)

## Training set

In [14]:
PandasTools.AddMoleculeColumnToFrame(data_train, "canonical_smiles", "Molecule")
HTML(data_train.head().to_html())

,molecule_chembl_id,canonical_smiles,pchembl_value_mean,inchi,Molecule
0,CHEMBL1767046,CNC(=O)/C(CCCCC[C@H](NCc1ccc(OC)cc1)C(=O)Nc1ccccc1)=N\O,4.02,"InChI=1S/C24H32N4O4/c1-25-23(29)22(28-31)12-8-4-7-11-21(24(30)27-19-9-5-3-6-10-19)26-17-18-13-15-20(32-2)16-14-18/h3,5-6,9-10,13-16,21,26,31H,4,7-8,11-12,17H2,1-2H3,(H,25,29)(H,27,30)/b28-22-/t21-/m0/s1",<rdkit.Chem.rdchem.Mol object at 0x000001308F2CCD60>
1,CHEMBL1469,O=C(O)CCCc1ccccc1,4.03,"InChI=1S/C10H12O2/c11-10(12)8-4-7-9-5-2-1-3-6-9/h1-3,5-6H,4,7-8H2,(H,11,12)",<rdkit.Chem.rdchem.Mol object at 0x000001308F2E83C0>
2,CHEMBL3605493,Nc1ccccc1NC(=O)c1ccc(CSC2=N[C@@H](Cc3ccccc3)CO2)cc1,4.06,"InChI=1S/C24H23N3O2S/c25-21-8-4-5-9-22(21)27-23(28)19-12-10-18(11-13-19)16-30-24-26-20(15-29-24)14-17-6-2-1-3-7-17/h1-13,20H,14-16,25H2,(H,27,28)/t20-/m0/s1",<rdkit.Chem.rdchem.Mol object at 0x000001308F2E84A0>
3,CHEMBL1767047,CNC(=O)/C(CCCCC[C@@H](C(=O)Nc1ccccc1)N(Cc1ccccc1)Cc1ccccc1)=N\O,4.06,"InChI=1S/C30H36N4O3/c1-31-29(35)27(33-37)20-12-5-13-21-28(30(36)32-26-18-10-4-11-19-26)34(22-24-14-6-2-7-15-24)23-25-16-8-3-9-17-25/h2-4,6-11,14-19,28,37H,5,12-13,20-23H2,1H3,(H,31,35)(H,32,36)/b33-27-/t28-/m0/s1",<rdkit.Chem.rdchem.Mol object at 0x000001308F2E8430>
4,CHEMBL3605499,Nc1ccccc1NC(=O)c1ccc(CNC2=N[C@@H](c3ccc(O)cc3)CO2)cc1,4.08,"InChI=1S/C23H22N4O3/c24-19-3-1-2-4-20(19)26-22(29)17-7-5-15(6-8-17)13-25-23-27-21(14-30-23)16-9-11-18(28)12-10-16/h1-12,21,28H,13-14,24H2,(H,25,27)(H,26,29)/t21-/m1/s1",<rdkit.Chem.rdchem.Mol object at 0x000001308F2E8350>


In [15]:
sdf_properties = [col for col in data_train.columns if col != "Molecule"]
PandasTools.WriteSDF(
    data_train,
    "HDAC8_train.sdf",
    molColName="Molecule",
    properties=sdf_properties,
)